In [1]:
import psutil
import torch
import os
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import time

print("="*50)
print("🔍 CONFIGURACIÓN DEL SISTEMA")
print("="*50)
print(f"CPU Físicos: {psutil.cpu_count(logical=False)}")
print(f"CPU Lógicos    : {psutil.cpu_count(logical=True)}")
print(f"RAM Total      : {psutil.virtual_memory().total / 1e9:.2f} GB")
print(f"RAM Disponible : {psutil.virtual_memory().available / 1e9:.2f} GB")
print(f"PyTorch Threads: {torch.get_num_threads()}")
print(f"Contenedor     : {'SI' if os.path.exists('/.dockerenv') else 'NO'}")
print("="*50)

class SystemMonitor:
    def __init__(self):
        self.cpu_history = []
        self.ram_history = []
        self.timestamps = []

    def snapshot(self):
        self.cpu_history.append(psutil.cpu_percent(interval=0.1))
        self.ram_history.append(psutil.virtual_memory().percent)
        self.timestamps.append(time.time())

    def plot(self):
        plt.figure(figsize=(12,4))

        plt.subplot(1,2,1)
        plt.plot(self.cpu_history)
        plt.title("CPU Usage (%)")

        plt.subplot(1,2,2)
        plt.plot(self.ram_history)
        plt.title("RAM Usage (%)")

        plt.tight_layout()
        plt.show()

monitor = SystemMonitor()

🔍 CONFIGURACIÓN DEL SISTEMA
CPU Físicos: 8
CPU Lógicos    : 16
RAM Total      : 7.98 GB
RAM Disponible : 6.43 GB
PyTorch Threads: 8
Contenedor     : SI


In [2]:
import numpy as np
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
import torch.optim as optim

class ExtremeStressDataset(Dataset):
    def __init__(self, num_samples=20, img_size=1000):
        self.num_samples = num_samples
        self.img_size = img_size

        print(
            f"Generando {num_samples} imágenes "
            f"de {img_size}x{img_size}"
        )

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):

        img = np.random.randn(
            3,
            self.img_size,
            self.img_size
        ).astype(np.float32)

        label = np.random.randint(0,10)

        if idx % 5 == 0:
            monitor.snapshot()

        return (
            torch.from_numpy(img),
            torch.tensor(label)
        )

IMG_SIZE = 1000
BATCH_SIZE = 1

dataset = ExtremeStressDataset(
    num_samples=20,
    img_size=IMG_SIZE
)

dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)

Generando 20 imágenes de 1000x1000


In [3]:
class ResourceIntensiveModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3,64,3,stride=2,padding=1),
            nn.ReLU(),

            nn.Conv2d(64,128,3,stride=2,padding=1),
            nn.ReLU(),

            nn.Conv2d(128,256,3,stride=2,padding=1),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d((32,32))
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(256*32*32,4096),
            nn.ReLU(),

            nn.Linear(4096,4096),
            nn.ReLU(),

            nn.Linear(4096,10)
        )

    def forward(self,x):
        x = self.conv_layers(x)
        x = x.view(x.size(0),-1)
        return self.fc_layers(x)

model = ResourceIntensiveModel()

total_params = sum(
    p.numel()
    for p in model.parameters()
)

print("Parámetros:", total_params)

Parámetros: 1090939018


In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

epochs = 2

for epoch in range(epochs):

    print(
        f"Epoch {epoch+1}/{epochs}"
    )

    for batch_idx, (data,target) in enumerate(dataloader):

        optimizer.zero_grad()

        output = model(data)

        loss = criterion(
            output,
            target
        )

        loss.backward()

        optimizer.step()

        if batch_idx % 2 == 0:

            mem = psutil.Process().memory_info()

            print(
                f"Batch {batch_idx}"
            )

            print(
                f"RAM proceso: "
                f"{mem.rss/1e9:.2f} GB"
            )

            print(
                f"CPU: "
                f"{psutil.cpu_percent()}%"
            )

        monitor.snapshot()

print("Entrenamiento terminado")

monitor.plot()

Epoch 1/2
